In [7]:
import os
import glob
import math
import sys
import bs4
import re
import csv
from decimal import Decimal, ROUND_DOWN
from io import StringIO
import pandas as pd
import xml.etree.ElementTree as ET

In [8]:
networkPropertiesPath = "./Inputs/Network Properties.html"
pf40FilePath = "./WorkingFolder/Inputs/PF40/"
pf70FilePath = "./WorkingFolder/Inputs/PF70/"
newFilePath = "WorkingFolder/Outputs"
pf525_template_path = "./Inputs/Templates/PF525Template.csv"
out_pf5_dir = "./Outputs/PF5 Files"
out_csv_dir = "WorkingFolder/Outputs/CSV Files"

In [9]:
#Variable Definition
pf40Template = pd.read_csv('./Inputs/Templates/PF40Template.csv')
pf70Template = pd.read_csv('./Inputs/Templates/PF70Template.csv')
networkproperties = open('./Inputs/Network Properties.html')
networkHTML = bs4.BeautifulSoup(networkproperties.read(), 'html.parser')

In [10]:
def remove_units(Value):
    if " " in str(Value):
        return Value.split()[0]
    return None

def pf40_make_dataframe(Number):
    DriveDf['Port'] = pf40Template['Port']
    DriveDf['#'] = pf40Template['#']
    DriveDf['Name'] = pf40Template['Name']
    DriveDf['Units'] = pf40Template['Units']
    DriveDf['Default'] = pf40Template['Default']
    DriveDf['Min'] = pf40Template['Min']
    DriveDf['Max'] = pf40Template['Max']

    
    for i in range(0, DriveDf.index.stop - 1):
    
        #conditional establishing
        check1 = str(DriveDf.at[i, 'Name']) in str(d[Number]['Name'].values)
        check2 = str(DriveDf.at[i, 'Port']) == '0'
        allOkay = check1 and check2
        if allOkay:
            NewValue = get_new_value(i, Number)
            #Look for missing units, to decide if I truncate off the Units
            if str(DriveDf.at[i, 'Units']) != 'nan':
                DriveDf.at[i, 'Value'] = remove_units(str(NewValue))
            else:
                DriveDf.at[i, 'Value'] = str(NewValue)
        else:
            #catch for if the name doesn't exist, or if it is on a different port
            DriveDf.at[i, 'Value'] = str(DriveDf.at[i, 'Default'])
    return DriveDf

def pf70_make_dataframe(Number):
    DriveDf['Port'] = pf70Template['Port']
    DriveDf['#'] = pf70Template['#']
    DriveDf['Name'] = pf70Template['Name']
    DriveDf['Units'] = pf70Template['Units']
    DriveDf['Default'] = pf70Template['Default']
    DriveDf['Min'] = pf70Template['Min']
    DriveDf['Max'] = pf70Template['Max']

    
    for i in range(0, DriveDf.index.stop - 1):
    
        #conditional establishing
        check1 = str(DriveDf.at[i, 'Name']) in str(d[Number]['Name'].values)
        check2 = str(DriveDf.at[i, 'Port']) == '0'
        allOkay = check1 and check2
        if allOkay:
            NewValue = get_new_value(i, Number)
            #Look for missing units, to decide if I truncate off the Units
            if str(DriveDf.at[i, 'Units']) != 'nan':
                DriveDf.at[i, 'Value'] = remove_units(str(NewValue))
            else:
                DriveDf.at[i, 'Value'] = str(NewValue)
        else:
            #catch for if the name doesn't exist, or if it is on a different port
            DriveDf.at[i, 'Value'] = str(DriveDf.at[i, 'Default'])
    return DriveDf
    
def get_new_value(iteration, Number):
    conditional = d[Number]['Name'] == DriveDf.at[iteration, 'Name']
    location = d[Number].loc[conditional]
    return next(iter(location['Value'].values))

def IPConfig(Address, Number):
    FirstOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'IP Addr Cfg 1'].index[0]
    SecondOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'IP Addr Cfg 2'].index[0]
    ThirdOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'IP Addr Cfg 3'].index[0]
    FourthOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'IP Addr Cfg 4'].index[0]
    DriveParameters[Number].at[FirstOctetIndex, 'Value'] = Address.split(".")[0]
    DriveParameters[Number].at[SecondOctetIndex, 'Value'] = Address.split(".")[1]
    DriveParameters[Number].at[ThirdOctetIndex, 'Value'] = Address.split(".")[2]
    DriveParameters[Number].at[FourthOctetIndex, 'Value'] = Address.split(".")[3]

def SubnetConfig(Address, Number):
    FirstOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Subnet Cfg 1'].index[0]
    SecondOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Subnet Cfg 2'].index[0]
    ThirdOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Subnet Cfg 3'].index[0]
    FourthOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Subnet Cfg 4'].index[0]
    DriveParameters[Number].at[FirstOctetIndex, 'Value'] = Address.split(".")[0]
    DriveParameters[Number].at[SecondOctetIndex, 'Value'] = Address.split(".")[1]
    DriveParameters[Number].at[ThirdOctetIndex, 'Value'] = Address.split(".")[2]
    DriveParameters[Number].at[FourthOctetIndex, 'Value'] = Address.split(".")[3]

def GateWayConfig(Address, Number):
    FirstOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Gateway Cfg 1'].index[0]
    SecondOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Gateway Cfg 2'].index[0]
    ThirdOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Gateway Cfg 3'].index[0]
    FourthOctetIndex = DriveParameters[Number].loc[DriveParameters[Number]['Name'] == 'Gateway Cfg 4'].index[0]
    DriveParameters[Number].at[FirstOctetIndex, 'Value'] = Address.split(".")[0]
    DriveParameters[Number].at[SecondOctetIndex, 'Value'] = Address.split(".")[1]
    DriveParameters[Number].at[ThirdOctetIndex, 'Value'] = Address.split(".")[2]
    DriveParameters[Number].at[FourthOctetIndex, 'Value'] = Address.split(".")[3]

In [11]:
def normalize_str(s):
    """Normalize values for reliable string comparisons (handles fancy quotes & HTML entities)."""
    if s is None or (isinstance(s, float) and pd.isna(s)):
        return None
    t = str(s)
    # Replace curly quotes and special punctuation with ASCII counterparts
    t = (t.replace('\u201c', '"').replace('\u201d', '"')
           .replace('\u2018', "'").replace('\u2019', "'")
           .replace('\u2013', '-').replace('\u2014', '-')
           .replace('&amp;', '&'))
    # Normalize whitespace and case for comparison
    return ' '.join(t.split()).strip().lower()
    
def decimal_places(s):
    """Return the number of decimal places in a numeric string (best-effort)."""
    if s is None:
        return 0
    s = str(s).strip()
    if '.' in s:
        return len(s.split('.', 1)[1])
    return 0
    
def trunc_mul(val_str, factor):
    """
    Multiply decimal string by factor and truncate toward zero (PowerShell's [math]::Truncate).
    Returns int.
    """
    if val_str is None or str(val_str).strip() == '':
        return None
    d = Decimal(str(val_str).strip())
    prod = d * Decimal(str(factor))
    # Truncate toward zero
    if prod >= 0:
        return int(prod.to_integral_value(rounding=ROUND_DOWN))
    else:
        return -int((-prod).to_integral_value(rounding=ROUND_DOWN))
        
def first_match_value(df, name):
    """Return the first Value for the given Name in df (or None)."""
    sel = df.loc[df['Name'] == name]
    if sel.empty:
        return None
    v = sel.iloc[0]['Value']
    if pd.isna(v):
        return None
    return str(v)

def set_value(df, name, value):
    """Set the Value for the first matching row by Name (no-op if not found)."""
    idx = df.index[df['Name'] == name]
    if len(idx) > 0:
        df.at[idx[0], 'Value'] = str(value)

def map_string_to_index(df, name, options):
    """
    Map df['Value'] at row 'name' to the index of 'options' (string match, normalized).
    Mirrors PS: [Array]::IndexOf(...) -> returns -1 if not found; Abs(-1) == 1 (bizarre).
    We'll follow that behavior to be faithful.
    """
    v = first_match_value(df, name)
    if v is None:
        return
    norm_v = normalize_str(v)
    # Normalize options for matching
    norm_options = [normalize_str(o) for o in options]
    try:
        idx = norm_options.index(norm_v)
    except ValueError:
        idx = -1  # not found (PS behavior)
    idx_final = abs(idx)  # PowerShell used Abs(IndexOf)
    set_value(df, name, idx_final)

def create_dataframe(csv_path):
    """
    Load CSV and ensure the expected columns exist: Port, #, Name, Value, Units, Default, Min, Max.
    """
    df = pd.read_csv(csv_path)
    # Ensure schema and ordering
    needed = ["Port", "#", "Name", "Value", "Units", "Default", "Min", "Max"]
    for col in needed:
        if col not in df.columns:
            df[col] = ""
    df = df[needed]
    return df
def pf40_search_and_replace(pf525_df, pf40_df):
    """
    For each PF525 row, find pf40 row with matching Name.
    If found and Value non-null:
       - if 1 or 2 decimals -> truncate(Value * 10 or * 100)
       - else copy literal
    If not found:
       - fall back to Default (with same decimal-scaling logic applied to Default)
    """
    for i, row in pf525_df.iterrows():
        name = row['Name']
        pf40_match = pf40_df.loc[pf40_df['Name'] == name].sort_values(by="#", kind='stable')
        if not pf40_match.empty:
            val = pf40_match.iloc[0]['Value']
            if not pd.isna(val) and str(val).strip() != '':
                dp = decimal_places(val)
                if dp == 1:
                    pf525_df.at[i, 'Value'] = str(trunc_mul(val, 10))
                elif dp == 2:
                    pf525_df.at[i, 'Value'] = str(trunc_mul(val, 100))
                else:
                    pf525_df.at[i, 'Value'] = str(val)
                continue  # done
        # If not found or null => use Default (respect decimals similarly)
        default_val = row['Default']
        dpd = decimal_places(default_val)
        if dpd == 1:
            pf525_df.at[i, 'Value'] = str(trunc_mul(default_val, 10))
        elif dpd == 2:
            pf525_df.at[i, 'Value'] = str(trunc_mul(default_val, 100))
        else:
            pf525_df.at[i, 'Value'] = str(default_val)

def pf70_search_and_replace(pf525_df, pf70_df):
    """
    For each PF525 row, find PF70 row with matching Name.
    If found and Value non-null:
       - if 1 or 2 decimals -> truncate(Value * 10 or * 100)
       - else copy literal
    If not found:
       - fall back to Default (with same decimal-scaling logic applied to Default)
    """
    for i, row in pf525_df.iterrows():
        name = row['Name']
        pf70_match = pf70_df.loc[pf70_df['Name'] == name].sort_values(by="#", kind='stable')
        if not pf70_match.empty:
            val = pf70_match.iloc[0]['Value']
            if not pd.isna(val) and str(val).strip() != '':
                dp = decimal_places(val)
                if dp == 1:
                    pf525_df.at[i, 'Value'] = str(trunc_mul(val, 10))
                elif dp == 2:
                    pf525_df.at[i, 'Value'] = str(trunc_mul(val, 100))
                else:
                    pf525_df.at[i, 'Value'] = str(val)
                continue  # done
        # If not found or null => use Default (respect decimals similarly)
        default_val = row['Default']
        dpd = decimal_places(default_val)
        if dpd == 1:
            pf525_df.at[i, 'Value'] = str(trunc_mul(default_val, 10))
        elif dpd == 2:
            pf525_df.at[i, 'Value'] = str(trunc_mul(default_val, 100))
        else:
            pf525_df.at[i, 'Value'] = str(default_val)
            
def replace_read_only(pf525_df):
    """
    Force Value = Default for read-only parameters listed in your PS function Replace-Read-Only.
    """
    ro_names = [
        'Drive Status','Process Display','Process Fract','Control Source','Contrl In Status',
        'Output RPM', 'Output Speed', 'Power Saved', 'Average Power', 'Energy Saved',
        'Accum kWh Sav', 'Accum Cost Sav', 'Accum CO2 Sav', 'Control Temp'
    ]
    for nm in ro_names:
        defv = first_match_value(pf525_df, nm)  # this returns Value; we need Default
        # fetch Default separately
        sel = pf525_df.loc[pf525_df['Name'] == nm]
        if not sel.empty:
            default_val = sel.iloc[0]['Default']
            set_value(pf525_df, nm, default_val)

def pf40_name_switch_params(pf525_df, pf40_df):
    """
    Handle parameters that are named differently or require specific transforms / constants.
    Mirrors your Name-Switch-Params function plus switch statements.
    """
    # Scale/move basic parameters
    motor_np_fla = first_match_value(pf40_df, 'Motor NP FLA')
    if motor_np_fla is not None:
        set_value(pf525_df, 'Motor NP FLA', trunc_mul(motor_np_fla, 10))
        set_value(pf525_df, 'Motor OL Current', trunc_mul(Decimal(motor_np_fla) * Decimal('1.25'), 10))

    motor_poles = first_match_value(pf40_df, 'Motor Poles')
    if motor_poles is not None:
        set_value(pf525_df, 'Motor NP Poles', motor_poles)

    min_speed = first_match_value(pf40_df, 'Minimum Freq')
    if min_speed is not None:
        set_value(pf525_df, 'Minimum Freq', trunc_mul(min_speed, 100))

    max_speed = first_match_value(pf40_df, 'Maximum Freq')
    if max_speed is not None:
        set_value(pf525_df, 'Maximum Freq', trunc_mul(max_speed, 100))

    # Constants
    set_value(pf525_df, 'Start Source 1', "5")
    set_value(pf525_df, 'Speed Reference1', "15")

    # IP / Subnet / Gateway mapping
    for i in range(1, 5):
        set_value(pf525_df, f'EN IP Addr Cfg {i}', first_match_value(pf40_df, f'IP Addr Cfg {i}') or '')
        set_value(pf525_df, f'EN Subnet Cfg {i}', first_match_value(pf40_df, f'Subnet Cfg {i}') or '')
        set_value(pf525_df, f'EN Gateway Cfg {i}', first_match_value(pf40_df, f'Gateway Cfg {i}') or '')

    # PWM Frequency (PS multiplies by 10, no Truncate invoked explicitly there; we still truncate to be safe)
    pwm = first_match_value(pf40_df, 'PWM Frequency')
    if pwm is not None and pwm != '':
        set_value(pf525_df, 'PWM Frequency', trunc_mul(pwm, 10))

    # Switches from pf40 -> PF525
    # Motor Cntl Sel -> Torque Perf Mode
    mcs = normalize_str(first_match_value(pf40_df, "Torque Perf Mode"))
    if mcs is not None:
        mapping = {
            normalize_str("Sensrls Vect"): "1",
            normalize_str("SV Economize"): "2",
            normalize_str("Custom V/Hz"): "0",
            normalize_str("Fan/Pmp V/Hz"): "0",
            normalize_str("FVC Vector"): "3",
        }
        set_value(pf525_df, 'Torque Perf Mode', mapping.get(mcs, normalize_str(first_match_value(pf525_df, 'Torque Perf Mode') or '')))

    # Stop/Brk Mode A -> Stop Mode
    sbm = normalize_str(first_match_value(pf40_df, "Stop Mode"))
    if sbm is not None:
        mapping = {
            normalize_str("Coast"): "1",
            normalize_str("Ramp"): "0",
            normalize_str("Ramp to Hold"): "9",
            normalize_str("DC Brake"): "6",
            normalize_str("Fast Brake"): "6",
        }
        set_value(pf525_df, 'Stop Mode', mapping.get(sbm, normalize_str(first_match_value(pf525_df, 'Stop Mode') or '')))

    # Comm Flt Action -> Comm Loss Action
    cfa = normalize_str(first_match_value(pf40_df, "Comm Loss Action"))
    if cfa is not None:
        mapping = {
            normalize_str("Fault"): "0",
            normalize_str("Stop"): "1",
            normalize_str("Zero Data"): "1",  # per your PS
            normalize_str("Hold Last"): "3",
            normalize_str("Send Flt Cfg"): "0",
        }
        set_value(pf525_df, 'Comm Loss Action', mapping.get(cfa, normalize_str(first_match_value(pf525_df, 'Comm Loss Action') or '')))

    # BootP -> EN Addr Sel
    bootp = normalize_str(first_match_value(pf40_df, "BootP"))
    if bootp is not None:
        mapping = {
            normalize_str("Disabled"): "1",
            normalize_str("Enabled"): "2",
        }
        set_value(pf525_df, 'EN Addr Sel', mapping.get(bootp, normalize_str(first_match_value(pf525_df, 'EN Addr Sel') or '')))

    # Autotune (pf40) -> Autotune (PF525) numeric mode
    at = normalize_str(first_match_value(pf40_df, "Autotune"))
    if at is not None:
        mapping = {
            normalize_str("Ready"): "0",
            normalize_str("Static Tune"): "1",
            normalize_str("Rotate Tune"): "2",
            normalize_str("Calculate"): "0",
        }
        set_value(pf525_df, 'Autotune', mapping.get(at, normalize_str(first_match_value(pf525_df, 'Autotune') or '')))

def pf70_name_switch_params(pf525_df, pf70_df):
    """
    Handle parameters that are named differently or require specific transforms / constants.
    Mirrors your Name-Switch-Params function plus switch statements.
    """
    # Scale/move basic parameters
    motor_np_fla = first_match_value(pf70_df, 'Motor NP FLA')
    if motor_np_fla is not None:
        set_value(pf525_df, 'Motor NP FLA', trunc_mul(motor_np_fla, 10))
        set_value(pf525_df, 'Motor OL Current', trunc_mul(Decimal(motor_np_fla) * Decimal('1.25'), 10))

    motor_poles = first_match_value(pf70_df, 'Motor Poles')
    if motor_poles is not None:
        set_value(pf525_df, 'Motor NP Poles', motor_poles)

    min_speed = first_match_value(pf70_df, 'Minimum Speed')
    if min_speed is not None:
        set_value(pf525_df, 'Minimum Freq', trunc_mul(min_speed, 100))

    max_speed = first_match_value(pf70_df, 'Maximum Speed')
    if max_speed is not None:
        set_value(pf525_df, 'Maximum Freq', trunc_mul(max_speed, 100))

    # Constants
    set_value(pf525_df, 'Start Source 1', "5")
    set_value(pf525_df, 'Speed Reference1', "15")

    # IP / Subnet / Gateway mapping
    for i in range(1, 5):
        set_value(pf525_df, f'EN IP Addr Cfg {i}', first_match_value(pf70_df, f'IP Addr Cfg {i}') or '')
        set_value(pf525_df, f'EN Subnet Cfg {i}', first_match_value(pf70_df, f'Subnet Cfg {i}') or '')
        set_value(pf525_df, f'EN Gateway Cfg {i}', first_match_value(pf70_df, f'Gateway Cfg {i}') or '')

    # PWM Frequency (PS multiplies by 10, no Truncate invoked explicitly there; we still truncate to be safe)
    pwm = first_match_value(pf70_df, 'PWM Frequency')
    if pwm is not None and pwm != '':
        set_value(pf525_df, 'PWM Frequency', trunc_mul(pwm, 10))

    # Switches from PF70 -> PF525
    # Motor Cntl Sel -> Torque Perf Mode
    mcs = normalize_str(first_match_value(pf70_df, "Motor Cntl Sel"))
    if mcs is not None:
        mapping = {
            normalize_str("Sensrls Vect"): "1",
            normalize_str("SV Economize"): "2",
            normalize_str("Custom V/Hz"): "0",
            normalize_str("Fan/Pmp V/Hz"): "0",
            normalize_str("FVC Vector"): "3",
        }
        set_value(pf525_df, 'Torque Perf Mode', mapping.get(mcs, first_match_value(pf525_df, 'Torque Perf Mode') or ''))

    # Stop/Brk Mode A -> Stop Mode
    sbm = normalize_str(first_match_value(pf70_df, "Stop/Brk Mode A"))
    if sbm is not None:
        mapping = {
            normalize_str("Coast"): "1",
            normalize_str("Ramp"): "0",
            normalize_str("Ramp to Hold"): "9",
            normalize_str("DC Brake"): "6",
            normalize_str("Fast Brake"): "6",
        }
        set_value(pf525_df, 'Stop Mode', mapping.get(sbm, first_match_value(pf525_df, 'Stop Mode') or ''))

    # Comm Flt Action -> Comm Loss Action
    cfa = normalize_str(first_match_value(pf70_df, "Comm Flt Action"))
    if cfa is not None:
        mapping = {
            normalize_str("Fault"): "0",
            normalize_str("Stop"): "1",
            normalize_str("Zero Data"): "1",  # per your PS
            normalize_str("Hold Last"): "3",
            normalize_str("Send Flt Cfg"): "0",
        }
        set_value(pf525_df, 'Comm Loss Action', mapping.get(cfa, first_match_value(pf525_df, 'Comm Loss Action') or ''))

    # BootP -> EN Addr Sel
    bootp = normalize_str(first_match_value(pf70_df, "BootP"))
    if bootp is not None:
        mapping = {
            normalize_str("Disabled"): "1",
            normalize_str("Enabled"): "2",
        }
        set_value(pf525_df, 'EN Addr Sel', mapping.get(bootp, first_match_value(pf525_df, 'EN Addr Sel') or ''))

    # Autotune (PF70) -> Autotune (PF525) numeric mode
    at = normalize_str(first_match_value(pf70_df, "Autotune"))
    if at is not None:
        mapping = {
            normalize_str("Ready"): "0",
            normalize_str("Static Tune"): "1",
            normalize_str("Rotate Tune"): "2",
            normalize_str("Calculate"): "0",
        }
        set_value(pf525_df, 'Autotune', mapping.get(at, first_match_value(pf525_df, 'Autotune') or ''))

def map_values_to_numbers(pf525_df):
    """
    Convert PF525 string parameters to the expected numeric codes by index or mapping.
    This mirrors your Map-Values-to-Numbers function (arrays/switches). To keep it maintainable,
    it's expressed as a table of (parameter, options) processed in a loop, plus some special cases.
    """
    # 1) Simple option-array mappings (index-of)
    simple_maps = [
        ("Autotune", ["Ready", "Static Tune", "Rotary Tune"]),  # spelling in PS had some variants
        ("Language", ['Not Used', 'English', 'Français', 'Español', 'Italiano', 'Deutsch',
                      'Japanese', 'Português', 'Chinese', 'Reserved', 'Reserved',
                      'Korean', 'Polish', 'Reserved', 'Turkish', 'Czech']),
        ("Start Source 2", ['Keypad', 'DigIn TrmBlk', 'Serial/DSI', 'Network Opt', 'EtherNet/IP']),
        ("Start Source 3", ['Keypad', 'DigIn TrmBlk', 'Serial/DSI', 'Network Opt', 'EtherNet/IP']),
        ("Speed Reference2", ['Drive Pot', 'Keypad Freq', 'Serial/DSI', 'Network Opt', '0-10V Input',
                              '4-20mA Input', 'Preset Freq', 'Anlg In Mult', 'MOP', 'Pulse Input',
                              'PID1 Output', 'PID2 Output', 'Step Logic', 'Encoder', 'EtherNet/IP', 'Positioning']),
        ("Speed Reference3", ['Drive Pot', 'Keypad Freq', 'Serial/DSI', 'Network Opt', '0-10V Input',
                              '4-20mA Input', 'Preset Freq', 'Anlg In Mult', 'MOP', 'Pulse Input',
                              'PID1 Output', 'PID2 Output', 'Step Logic', 'Encoder', 'EtherNet/IP', 'Positioning']),
        ("DigIn TermBlk 02", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("DigIn TermBlk 03", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("DigIn TermBlk 05", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("DigIn TermBlk 06", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("DigIn TermBlk 07", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("DigIn TermBlk 08", ['Not Used', 'Speed Ref 2', 'Speed Ref 3', 'Start Src 2', 'Start Src 3',
                              'Preset Freq', 'Jog', 'Jog Forward', 'Jog Reverse', 'Acc/Decc Sel2',
                              'Aux Fault', 'Clear Fault', 'RampStop,CF', 'CoastStop,CF', 'DCInjStop,CF', 'MOP Up',
                              'MOP Down', 'Timer Start', 'Counter In', 'Reset Timer', 'Reset Countr', 'Rset Tim&Cnt',
                              'Logic In 1', 'Logic In 2', 'Current Lmt2', 'Anlg Invert', 'EM Brk Rlse', 'Acc/Dec Sel3',
                              'Precharge En', 'Inertia Dcel', 'Sync Enable', 'Traverse Dis', 'Home Limit', 'Find Home',
                              'Hold Step', 'Pos Redefine', 'Force DC', 'Damper Input', 'Purge', 'Freeze-Fire',
                              'SW Enable', 'SherPin1 Dis', 'Reserver', 'Reserved', 'Reserved', 'Reserved',
                              '2-Wire FWD', '3-Wire Start', '2-Wire REV', '3-Wire Dir', 'Pulse Train']),
        ("2-Wire Mode", ['Edge Trigger', 'Level Sense', 'Hi-Spd Edge', 'Momentary']),
        ("10V Bipolar Enbl", ['Uni-Polar In', 'Bi-Polar In', 'Hi-Spd Edge', 'Momentary']),
        ("Opto Out1 Sel", ['Ready/Fault', 'At Frequency', 'MotorRunning', 'Reverse', 'Motor Overld', 'Ramp Reg',
                           'Above Freq', 'Above Cur', 'Above DCVolt', 'Retries Exst', 'Above Anlg V', 'Above PF Ang',
                           'Anlg In Loss', 'ParamControl', 'NonRec Fault', 'EM Brk Cntrl', 'Thermal OL',
                           'Amb OverTemp', 'Local Active', 'Comm Loss', 'Logic In 1', 'Logic In 2', 'Logic 1 & 2',
                           'Logic 1 or 2', 'StpLogic Out', 'Timer Out', 'Counter Out', 'At Position', 'At Home',
                           'Safe-Off', 'SafeTqPermit', 'AutoRst Ctdn']),
        ("Opto Out2 Sel", ['Ready/Fault', 'At Frequency', 'MotorRunning', 'Reverse', 'Motor Overld', 'Ramp Reg',
                           'Above Freq', 'Above Cur', 'Above DCVolt', 'Retries Exst', 'Above Anlg V', 'Above PF Ang',
                           'Anlg In Loss', 'ParamControl', 'NonRec Fault', 'EM Brk Cntrl', 'Thermal OL',
                           'Amb OverTemp', 'Local Active', 'Comm Loss', 'Logic In 1', 'Logic In 2', 'Logic 1 & 2',
                           'Logic 1 or 2', 'StpLogic Out', 'Timer Out', 'Counter Out', 'At Position', 'At Home',
                           'Safe-Off', 'SafeTqPermit', 'AutoRst Ctdn']),
        ("Relay Out1 Sel", ['Ready/Fault', 'At Frequency', 'MotorRunning', 'Reverse', 'Motor Overld', 'Ramp Reg',
                            'Above Freq', 'Above Cur', 'Above DCVolt', 'Retries Exst', 'Above Anlg V', 'Above PF Ang',
                            'Anlg In Loss', 'ParamControl', 'NonRec Fault', 'EM Brk Cntrl', 'Thermal OL',
                            'Amb OverTemp', 'Local Active', 'Comm Loss', 'Logic In 1', 'Logic In 2', 'Logic 1 & 2',
                            'Logic 1 or 2', 'StpLogic Out', 'Timer Out', 'Counter Out', 'At Position', 'At Home',
                            'Safe-Off', 'SafeTqPermit', 'AutoRst Ctdn']),
        ("Relay Out2 Sel", ['Ready/Fault', 'At Frequency', 'MotorRunning', 'Reverse', 'Motor Overld', 'Ramp Reg',
                            'Above Freq', 'Above Cur', 'Above DCVolt', 'Retries Exst', 'Above Anlg V', 'Above PF Ang',
                            'Anlg In Loss', 'ParamControl', 'NonRec Fault', 'EM Brk Cntrl', 'Thermal OL',
                            'Amb OverTemp', 'Local Active', 'Comm Loss', 'Logic In 1', 'Logic In 2', 'Logic 1 & 2',
                            'Logic 1 or 2', 'StpLogic Out', 'Timer Out', 'Counter Out', 'At Position', 'At Home',
                            'Safe-Off', 'SafeTqPermit', 'AutoRst Ctdn']),
        ("MOP Reset Sel", ['Zero MOP Ref', 'Save MOP Ref']),
        ("MOP Preload", ['No preload', 'Preload']),
        ("DB Resistor Sel", ['Disabled', 'Norml RA Res', 'NoProtection', '99"3...99% DutyCycle']),
        ("PID 1 Trim Sel", ['Disabled', 'TrimOn Pot', 'TrimOn Keypd', 'TrimOn DSI', 'TrimOn NetOp', 'TrimOn 0-10V',
                            'TrimOn 4-20', 'TrimOn Prset', 'TrimOn AnMlt', 'TrimOn MOP', 'TrimOn Pulse',
                            'TrimOn Slgic', 'TrimOn Encdr', 'TrimOn ENet']),
        ("PID 2 Trim Sel", ['Disabled', 'TrimOn Pot', 'TrimOn Keypd', 'TrimOn DSI', 'TrimOn NetOp', 'TrimOn 0-10V',
                            'TrimOn 4-20', 'TrimOn Prset', 'TrimOn AnMlt', 'TrimOn MOP', 'TrimOn Pulse',
                            'TrimOn Slgic', 'TrimOn Encdr', 'TrimOn ENet']),
        ("PID 1 Ref Sel", ['PID Setpoint', 'Drive Pot', 'Keypad Freq', 'Serial/DSI', 'Network Opt', '0-10V Input',
                           '4-20mA Input', 'Preset Freq', 'AnlgIn Multi', 'MOP Freq', 'Pulse Input', 'Step Logic',
                           'Encoder', 'EtherNet/IP']),
        ("PID 2 Ref Sel", ['PID Setpoint', 'Drive Pot', 'Keypad Freq', 'Serial/DSI', 'Network Opt', '0-10V Input',
                           '4-20mA Input', 'Preset Freq', 'AnlgIn Multi', 'MOP Freq', 'Pulse Input', 'Step Logic',
                           'Encoder', 'EtherNet/IP']),
        ("PID 1 Fdback Sel", ['0-10V Input', '4-20mA Input', 'Serial/DSI', 'Network Opt', 'Pulse Input', 'Encoder', 'EtherNet/IP']),
        ("PID 2 Fdback Sel", ['0-10V Input', '4-20mA Input', 'Serial/DSI', 'Network Opt', 'Pulse Input', 'Encoder', 'EtherNet/IP']),
        ("Stall Fault Time", ['60 Seconds', '120 Seconds', '240 Seconds', '360 Seconds', '480 Seconds', 'Flt Disabled']),
        ("Motor OL Select", ['No Derate', 'Min Derate', 'Max Derate']),
        ("Motor OL Ret", ['Reset', 'Save']),
        ("Drive OL Mode", ['Disabled', 'Reduce CLim', 'Reduce PWM', 'Both-PWM 1st']),
        ("Speed Reg Sel", ['Automatic', 'Manual']),
        ("PM Initial Sel", ['Align', 'HFI', 'Six Pulse']),
        ("Boost Select", ['Custom V/Hz', '30.0, VT', '35.0, VT', '40.0, VT', '45.0, VT', '0.0, no IR',
                          '2.5, CT', '2.5, CT', '5.0, CT', '7.5, CT', '10.0, CT', '12.5, CT', '15.0, CT', '17.5, CT', '20.0, CT']),
        ("Motor Fdbk Type", ['None', 'Pulse Train', 'Single Chan', 'Single Check', 'Quadrature', 'Quad Check']),
        ("Var PWM Disable", ['Enabled', 'Disabled']),
        ("Reverse Disable", ['Rev Enabled', 'Rev Disabled']),
        ("Flying Start En", ['Disabled', 'Enabled']),
        ("Compensation", ['Disabled', 'Electrical', 'Mechanical', 'Both']),
        ("Power Loss Mode", ['Coast', 'Decel']),
        ("Half Bus Enable", ['Disabled', 'Enabled']),
        ("Bus Reg Enable", ['Disabled', 'Enabled']),
        ("Program Lock Mod", ['Full Lock', 'Keypad Lock', 'Custom Only', 'KeyPd Custom']),
        ("Drv Ambient Sel", ['Normal', '55C', '60C', '60C +Fan Kit', '70C +Fan Kit']),
        ("Text Scroll", ['Off', 'Low Speed', 'Mid Speed', 'High Speed']),
        ("Out Phas Loss En", ['Disable', 'Enable']),
        ("Positioning Mode", ['Time Steps', 'Preset Input', 'Step Logic', 'Preset StpL', 'StpLogic-Lst']),
        ("Home Save", ['Home Reset', 'Home Saved']),
        ("Find Home Dir", ['Forward', 'Reverse']),
        ("PM Algor Sel", ['Algorithm 1', 'Algorithm 2']),
        ("EN Addr Src", ['Parameters', 'BOOTP']),
        ("EN Rate Act", ['No Link', '10Mbps Full', '10Mbps Half', '100Mbps Full', '100Mbps Half', 'Dup IP Addr', 'Disabled']),
        ("RdyBit Mode Cfg", ['Standard', 'Enhanced']),
        ("RdyBit Mode Act", ['Standard', 'Enhanced']),
        ("PID 1 Invert Err", ['Normal', 'Inverted']),
        ("PID 2 Invert Err", ['Normal', 'Inverted']),
        ("Start At PowerUp", ['Disabled', 'Enabled']),
        ("Flux Braking En", ['Disabled', 'Enabled']),
    ]
    for pname, options in simple_maps:
        map_string_to_index(pf525_df, pname, options)

    # 2) Switch-style exact mappings (string -> specific code)
    # Reset to Defalts (typo preserved from PS)
    val = normalize_str(first_match_value(pf525_df, "Reset to Defalts"))
    if val is not None:
        mapping = {
            normalize_str("Ready/Idle"): "0",
            normalize_str("Ready"): "0",
            normalize_str("Param Reset"): "1",
            normalize_str("Factory Rset"): "2",
            normalize_str("Power Reset"): "3",
            normalize_str("Module Reset"): "4",
        }
        set_value(pf525_df, "Reset to Defalts", mapping.get(val, "0"))

    # Display Param
    val = normalize_str(first_match_value(pf525_df, "Display Param"))
    if val is not None:
        mapping = {
            normalize_str("Keypad Disp"): "0",
            normalize_str("Output Freq"): "1",
        }
        set_value(pf525_df, "Display Param", mapping.get(val, "0"))

    # Reset Meters
    val = normalize_str(first_match_value(pf525_df, "Reset Meters"))
    if val is not None:
        mapping = {
            normalize_str("Ready/Idle"): "0",
            normalize_str("Ready"): "0",
            normalize_str("Reset Meters"): "1",
            normalize_str("Reset Time"): "2",
        }
        set_value(pf525_df, "Reset Meters", mapping.get(val, "0"))

    # Analog Out Sel
    val = normalize_str(first_match_value(pf525_df, "Analog Out Sel"))
    if val is not None:
        mapping = {
            normalize_str("OutFreq 0-10"): "0",
            normalize_str("OutCurr 0-10"): "1",
            normalize_str("OutVolt 0-10"): "2",
            normalize_str("OutPowr 0-10"): "3",
            normalize_str("OutTorq 0-10"): "4",
            normalize_str("TstData 0-10"): "5",
            normalize_str("Setpnt 0-10"): "6",
            normalize_str("DCVolt 0-10"): "7",
            normalize_str("OutFreq 0-20"): "8",
            normalize_str("OutCurr 0-20"): "9",
            normalize_str("OutVolt 0-20"): "10",
            normalize_str("OutPowr 0-20"): "11",
            normalize_str("OutTorq 0-20"): "12",
            normalize_str("TstData 0-20"): "13",
            normalize_str("Setpnt 0-20"): "14",
            normalize_str("DCVolt 0-20"): "15",
            normalize_str("OutFreq 4-20"): "16",
            normalize_str("OutCurr 4-20"): "17",
            normalize_str("OutVolt 4-20"): "18",
            normalize_str("OutPowr 4-20"): "19",
            normalize_str("OutTorq 4-20"): "20",
            normalize_str("TstData 4-20"): "21",
            normalize_str("Setpnt 4-20"): "22",
            normalize_str("DCVolt 4-20"): "23",
        }
        set_value(pf525_df, "Analog Out Sel", mapping.get(val, "0"))

    # Anlg In V Loss
    val = normalize_str(first_match_value(pf525_df, "Anlg In V Loss"))
    if val is not None:
        mapping = {
            normalize_str("Disabled"): "0",
            normalize_str("Fault (F29)"): "1",
            normalize_str("Stop"): "2",
            normalize_str("Zero Ref"): "3",
            normalize_str("Min Freq Ref"): "4",
            normalize_str("Max Freq Ref"): "5",
            normalize_str("Key Freq Ref"): "6",
            normalize_str("MOP Freq Ref"): "7",
            normalize_str("Continu Last"): "8",
        }
        set_value(pf525_df, "Anlg In V Loss", mapping.get(val, "0"))

    # Anlg In mA Loss
    val = normalize_str(first_match_value(pf525_df, "Anlg In mA Loss"))
    if val is not None:
        mapping = {
            normalize_str("Disabled"): "0",
            normalize_str("Fault (F29)"): "1",
            normalize_str("Stop"): "2",
            normalize_str("Zero Ref"): "3",
            normalize_str("Min Freq Ref"): "4",
            normalize_str("Max Freq Ref"): "5",
            normalize_str("Key Freq Ref"): "6",
            normalize_str("MOP Freq Ref"): "7",
            normalize_str("Continu Last"): "8",
        }
        set_value(pf525_df, "Anlg In mA Loss", mapping.get(val, "0"))

    # Sleep-Wake Sel
    val = normalize_str(first_match_value(pf525_df, "Sleep-Wake Sel"))
    if val is not None:
        mapping = {
            normalize_str("Disabled"): "0",
            normalize_str("0-10V Input"): "1",
            normalize_str("4-20mA Input"): "2",
            normalize_str("Command Freq"): "3",
        }
        set_value(pf525_df, "Sleep-Wake Sel", mapping.get(val, "0"))

    # Safety Open En
    val = normalize_str(first_match_value(pf525_df, "Safety Open En"))
    if val is not None:
        mapping = {
            normalize_str("FaultEnable"): "0",
            normalize_str("FaultDisable"): "1",
        }
        set_value(pf525_df, "Safety Open En", mapping.get(val, "0"))

    # SafetyFlt RstCfg
    val = normalize_str(first_match_value(pf525_df, "SafetyFlt RstCfg"))
    if val is not None:
        mapping = {
            normalize_str("PwrCycleRset"): "0",
            normalize_str("FltClr Reset"): "1",
        }
        set_value(pf525_df, "SafetyFlt RstCfg", mapping.get(val, "0"))

    # Comm Write Mode
    val = normalize_str(first_match_value(pf525_df, "Comm Write Mode"))
    if val is not None:
        mapping = {
            normalize_str("Save"): "0",
            normalize_str("RAM Only"): "1",
        }
        set_value(pf525_df, "Comm Write Mode", mapping.get(val, "0"))

    # Cmd Stat Select
    val = normalize_str(first_match_value(pf525_df, "Cmd Stat Select"))
    if val is not None:
        mapping = {
            normalize_str("Velocity"): "0",
            normalize_str("Position"): "1",
        }
        set_value(pf525_df, "Cmd Stat Select", mapping.get(val, "0"))

    # RS485 Data Rate
    val = normalize_str(first_match_value(pf525_df, "RS485 Data Rate"))
    if val is not None:
        mapping = {
            normalize_str("1200"): "0",
            normalize_str("2400"): "1",
            normalize_str("4800"): "2",
            normalize_str("9600"): "3",
            normalize_str("19,200"): "4",
            normalize_str("38,400"): "5",
        }
        set_value(pf525_df, "RS485 Data Rate", mapping.get(val, "3"))  # default to 9600 like common defaults

    # Comm Loss Action
    val = normalize_str(first_match_value(pf525_df, "Comm Loss Action"))
    if val is not None:
        mapping = {
            normalize_str("Fault"): "0",
            normalize_str("Coast Stop"): "1",
            normalize_str("Stop"): "2",
            normalize_str("Continu Last"): "3",
        }
        set_value(pf525_df, "Comm Loss Action", mapping.get(val, "2"))

    # RS485 Format
    val = normalize_str(first_match_value(pf525_df, "RS485 Format"))
    if val is not None:
        mapping = {
            normalize_str("RTU 8-N-1"): "0",
            normalize_str("RTU 8-E-1"): "1",
            normalize_str("RTU 8-O-1"): "2",
            normalize_str("RTU 8-N-2"): "3",
            normalize_str("RTU 8-E-2"): "4",
            normalize_str("RTU 8-O-2"): "5",
        }
        set_value(pf525_df, "RS485 Format", mapping.get(val, "0"))

    # EN Addr Sel
    val = normalize_str(first_match_value(pf525_df, "EN Addr Sel"))
    if val is not None:
        mapping = {
            normalize_str("Parameters"): "0",
            normalize_str("BOOTP"): "1",
        }
        set_value(pf525_df, "EN Addr Sel", mapping.get(val, "0"))

    # EN Rate Cfg
    val = normalize_str(first_match_value(pf525_df, "EN Rate Cfg"))
    if val is not None:
        mapping = {
            normalize_str("Auto Detect"): "0",
            normalize_str("Autodetect"): "0",
            normalize_str("Auto detect"): "0",
            normalize_str("10Mbps Full"): "1",
            normalize_str("10Mbps Half"): "2",
            normalize_str("100Mbps Full"): "3",
            normalize_str("100Mbps Half"): "4",
        }
        set_value(pf525_df, "EN Rate Cfg", mapping.get(val, "0"))

    # EN Comm Flt Actn
    val = normalize_str(first_match_value(pf525_df, "EN Comm Flt Actn"))
    if val is not None:
        mapping = {
            normalize_str("Fault"): "0",
            normalize_str("Stop"): "1",
            normalize_str("Zero Data"): "2",
            normalize_str("Hold Last"): "3",
            normalize_str("Send Flt Cfg"): "4",
        }
        set_value(pf525_df, "EN Comm Flt Actn", mapping.get(val, "1"))

    # EN Idle Flt Actn
    val = normalize_str(first_match_value(pf525_df, "EN Idle Flt Actn"))
    if val is not None:
        mapping = {
            normalize_str("Fault"): "0",
            normalize_str("Stop"): "1",
            normalize_str("Zero Data"): "2",
            normalize_str("Hold Last"): "3",
            normalize_str("Send Flt Cfg"): "4",
        }
        set_value(pf525_df, "EN Idle Flt Actn", mapping.get(val, "1"))

    # MultiDrv Sel
    val = normalize_str(first_match_value(pf525_df, "MultiDrv Sel"))
    if val is not None:
        mapping = {
            normalize_str("Disabled"): "0",
            normalize_str("Network Opt"): "1",
            normalize_str("EtherNet/IP"): "2",
        }
        set_value(pf525_df, "MultiDrv Sel", mapping.get(val, "0"))

    # DSI I/O Cfg
    val = normalize_str(first_match_value(pf525_df, "DSI I/O Cfg"))
    if val is not None:
        mapping = {
            normalize_str("Drive 0"): "0",
            normalize_str("Drive 0-1"): "1",
            normalize_str("Drive 0-2"): "2",
            normalize_str("Drive 0-3"): "3",
            normalize_str("Drive 0-4"): "4",
        }
        set_value(pf525_df, "DSI I/O Cfg", mapping.get(val, "0"))

    # Opto Out Logic
    val = normalize_str(first_match_value(pf525_df, "Opto Out Logic"))
    if val is not None:
        mapping = {
            normalize_str("1=NO / 2=NO"): "0",
            normalize_str("1=NC / 2=NO"): "1",
            normalize_str("1=NO / 2=NC"): "2",
            normalize_str("1=NC / 2=Nc"): "3",
        }
        set_value(pf525_df, "Opto Out Logic", mapping.get(val, "0"))

def dataframe_to_pf5(pf525_df, pf5_path):
    """
    Build the PF5 XML with non-default parameters only.
    The Drive WILL NOT TOLERATE TRYING TO PROGRAM READ ONLY PARAMETERS.
    """
    root = ET.Element("Node")
    drive = ET.SubElement(root, "Drive", Brand="1", Family="9", Config="196", MajorRev="7", MinorRev="1")
    params = ET.SubElement(drive, "Parameters")

    # Iterate rows; use Instance from '#' column (safer than PS row index)
    for _, r in pf525_df.iterrows():
        try:
            instance = int(str(r["#"]).strip())
        except Exception:
            continue
        if instance <= 29:
            continue
        if instance in {53, 71, 74, 78, 83, 101, 103, 201, 203, 205, 207, 209, 211, 213, 215, 483, 551}:
            continue
        if 360 <= instance <= 369:
            continue
        if 375 <= instance <= 394:
            continue
        if 604 <= instance <= 670:
            continue
        if 681 <= instance <= 683:
            continue
        if 686 <= instance <= 692:
            continue
        if instance >= 705:
            continue

        val = "" if pd.isna(r["Value"]) else str(r["Value"])
        default_val = "" if pd.isna(r["Default"]) else str(r["Default"])
        if val != default_val:
            p = ET.SubElement(params, "Parameter", Instance=str(instance))
            p.text = val

    # Write XML
    tree = ET.ElementTree(root)
    ET.indent(tree, space="  ", level=0)  # Python 3.9+
    tree.write(pf5_path, encoding="utf-8", xml_declaration=True)

In [14]:
def pf40_Conversion():
    devices = networkHTML.find_all('div', id='DEVICE_SEPARATOR')
    for div in devices:
        device_title = div.find('div', id='DEVICE_TITLE')
        if device_title:
            a_tag = device_title.find('a',  attrs={'name': re.compile('PowerFlex 40')})
            if a_tag:
                PowerFlex_List.append(div)
    
    DriveNumber = 0
    for Drive in PowerFlex_List:
            Drive_Names.append(Drive.find('div', id='DEVICE_TITLE').find('a')['name'])
            applet_table = StringIO(str(Drive.find('table', id = 'APPLET_TABLE_1')))
            if applet_table is not None:
                d[DriveNumber] = pd.read_html(applet_table)[0]
                DriveNumber = DriveNumber + 1
    for dataFrame in d:
        d[dataFrame].columns = csv_header
    for Drive in range(0, len(Drive_Names)):
        DriveParameters[f'{Drive}'] = pf40_make_dataframe(Drive)
    for Drive in range(0, len(DriveParameters)):
        IP = input(f"Please enter full IP for {Drive_Names[Drive]} \n")
        IPConfig(IP, str(Drive))
        Subnet = "255.255.255.0"
        SubnetConfig(Subnet, str(Drive))
        Gateway = input(f"Please enter full Gateway for {Drive_Names[Drive]} \n")
        GateWayConfig(Gateway, str(Drive))
    
        fileNameIndex = Drive - 1
        fileName = './WorkingFolder/Inputs/PF40/' + Drive_Names[fileNameIndex] + '.csv'
        DriveParameters.get(f"{Drive}").to_csv(fileName, index=False)

def pf70_Conversion():
    devices = networkHTML.find_all('div', id='DEVICE_SEPARATOR')
    for div in devices:
        device_title = div.find('div', id='DEVICE_TITLE')
        if device_title:
            a_tag = device_title.find('a',  attrs={'name': re.compile('PowerFlex 70')})
            if a_tag:
                PowerFlex_List.append(div)
    
    DriveNumber = 0
    for Drive in PowerFlex_List:
            Drive_Names.append(Drive.find('div', id='DEVICE_TITLE').find('a')['name'])
            applet_table = StringIO(str(Drive.find('table', id = 'APPLET_TABLE_1')))
            if applet_table is not None:
                d[DriveNumber] = pd.read_html(applet_table)[0]
                DriveNumber = DriveNumber + 1
    for dataFrame in d:
        d[dataFrame].columns = csv_header
    for Drive in range(0, len(Drive_Names)):
        DriveParameters[f'{Drive}'] = pf70_make_dataframe(Drive)
    for Drive in range(0, len(DriveParameters)):
        IP = input(f"Please enter full IP for {Drive_Names[Drive]} \n")
        IPConfig(IP, str(Drive))
        Subnet = "255.255.255.0"
        SubnetConfig(Subnet, str(Drive))
        Gateway = input(f"Please enter full Gateway for {Drive_Names[Drive]} \n")
        GateWayConfig(Gateway, str(Drive))
    
        fileNameIndex = Drive - 1
        fileName = './WorkingFolder/Inputs/PF70/' + Drive_Names[fileNameIndex] + '.csv'
        DriveParameters.get(f"{Drive}").to_csv(fileName, index=False)


def process_all_pf40():
    # Gather input pf40 CSV files
    pf40_files = glob.glob(os.path.join(pf40FilePath, "*.csv"))

    for fpath in pf40_files:
        base = os.path.splitext(os.path.basename(fpath))[0]
        print(f"Processing: {base}")

        pf525_df = create_dataframe(pf525_template_path).copy()
        pf40_df = create_dataframe(fpath).copy()

        # Execute workflow
        pf40_search_and_replace(pf525_df, pf40_df)
        replace_read_only(pf525_df)
        pf40_name_switch_params(pf525_df, pf40_df)
        map_values_to_numbers(pf525_df)

        # PF5 output
        pf5_out = os.path.join(out_pf5_dir, f"{base}.pf5")
        dataframe_to_pf5(pf525_df, pf5_out)

        # CSV output for CC import
        csv_out = os.path.join(out_csv_dir, f"{base}.csv")
        pf525_df.to_csv(csv_out, index=False)

        print(f"  -> PF5: {pf5_out}")
        print(f"  -> CSV: {csv_out}")

def process_all_pf70():
    # Gather input pf40 CSV files
    pf70_files = glob.glob(os.path.join(pf70FilePath, "*.csv"))

    for fpath in pf70_files:
        base = os.path.splitext(os.path.basename(fpath))[0]
        print(f"Processing: {base}")

        pf525_df = create_dataframe(pf525_template_path).copy()
        pf70_df = create_dataframe(fpath).copy()

        # Execute workflow
        pf70_search_and_replace(pf525_df, pf70_df)
        replace_read_only(pf525_df)
        pf70_name_switch_params(pf525_df, pf70_df)
        map_values_to_numbers(pf525_df)

        # PF5 output
        pf5_out = os.path.join(out_pf5_dir, f"{base}.pf5")
        dataframe_to_pf5(pf525_df, pf5_out)

        # CSV output for CC import
        csv_out = os.path.join(out_csv_dir, f"{base}.csv")
        pf525_df.to_csv(csv_out, index=False)

        print(f"  -> PF5: {pf5_out}")
        print(f"  -> CSV: {csv_out}")
if __name__ == "__main__":
    
    PowerFlex_List = []
    Drive_Names = []
    DriveParameters = {}
    d = {}
    csv_header = ['#', 'Name', 'Value']
    DriveDf = pd.DataFrame({
        'Port': pd.Series(dtype='str'),
        '#': pd.Series(dtype='str'),
        'Name': pd.Series(dtype='str'),
        'Value': pd.Series(dtype='str'),
        'Units': pd.Series(dtype='str'),
        'Default': pd.Series(dtype='str'),
        'Min': pd.Series(dtype='str'),
        'Max': pd.Series(dtype='str')
        })
    
    pf40_Conversion()
    
    PowerFlex_List = []
    Drive_Names = []
    DriveParameters = {}
    d = {}
    csv_header = ['#', 'Name', 'Value']
    DriveDf = pd.DataFrame({
        'Port': pd.Series(dtype='str'),
        '#': pd.Series(dtype='str'),
        'Name': pd.Series(dtype='str'),
        'Value': pd.Series(dtype='str'),
        'Units': pd.Series(dtype='str'),
        'Default': pd.Series(dtype='str'),
        'Min': pd.Series(dtype='str'),
        'Max': pd.Series(dtype='str')
        })
    
    pf70_Conversion()
    process_all_pf40()
    process_all_pf70()

Please enter full IP for Address 01,  PowerFlex 40 VFD01 Infeed Belt 
 192.168.1.2
Please enter full Gateway for Address 01,  PowerFlex 40 VFD01 Infeed Belt 
 255.255.255.0
Please enter full IP for Address 03,  PowerFlex 40 VFD03 Slat Devider 
 192.168.1.3
Please enter full Gateway for Address 03,  PowerFlex 40 VFD03 Slat Devider 
 255.255.255.0
Please enter full IP for Address 04,  PowerFlex 40 VFD04 Live Roller 
 192.168.1.4
Please enter full Gateway for Address 04,  PowerFlex 40 VFD04 Live Roller 
 255.255.255.0
Please enter full IP for Address 06,  PowerFlex 40 VFD05 Layer Pusher 
 192.168.1.6
Please enter full Gateway for Address 06,  PowerFlex 40 VFD05 Layer Pusher 
 255.255.255.0
Please enter full IP for Address 07,  07-PowerFlex 40 VFD09 Load Conveyor 
 192.168.1.7
Please enter full Gateway for Address 07,  07-PowerFlex 40 VFD09 Load Conveyor 
 255.255.255.0
Please enter full IP for Address 09,  09-PowerFlex 40 VFDF12 Discharge Conveyor 1 
 192.168.1.9
Please enter full Gateway

Processing: Address 01,  PowerFlex 40 VFD01 Infeed Belt
  -> PF5: ./Outputs/PF5 Files\Address 01,  PowerFlex 40 VFD01 Infeed Belt.pf5
  -> CSV: WorkingFolder/Outputs/CSV Files\Address 01,  PowerFlex 40 VFD01 Infeed Belt.csv
Processing: Address 03,  PowerFlex 40 VFD03 Slat Devider
  -> PF5: ./Outputs/PF5 Files\Address 03,  PowerFlex 40 VFD03 Slat Devider.pf5
  -> CSV: WorkingFolder/Outputs/CSV Files\Address 03,  PowerFlex 40 VFD03 Slat Devider.csv
Processing: Address 04,  PowerFlex 40 VFD04 Live Roller
  -> PF5: ./Outputs/PF5 Files\Address 04,  PowerFlex 40 VFD04 Live Roller.pf5
  -> CSV: WorkingFolder/Outputs/CSV Files\Address 04,  PowerFlex 40 VFD04 Live Roller.csv
Processing: Address 06,  PowerFlex 40 VFD05 Layer Pusher
  -> PF5: ./Outputs/PF5 Files\Address 06,  PowerFlex 40 VFD05 Layer Pusher.pf5
  -> CSV: WorkingFolder/Outputs/CSV Files\Address 06,  PowerFlex 40 VFD05 Layer Pusher.csv
Processing: Address 07,  07-PowerFlex 40 VFD09 Load Conveyor
  -> PF5: ./Outputs/PF5 Files\Address